In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score
)

print("Day 4 libraries imported successfully")

Day 4 libraries imported successfully


In [2]:
base_df = pd.read_csv("../data/frix_features_day1.csv")
graph_df = pd.read_csv("../data/frix_graph_features_day3.csv")

print("Base dataset:", base_df.shape)
print("Graph features:", graph_df.shape)


Base dataset: (1048575, 20)
Graph features: (1048575, 10)


In [3]:
graph_feature_cols = [
    "receiver_in_degree",
    "sender_out_degree",
    "receiver_total_amount",
    "receiver_avg_amount",
    "receiver_high_risk_txn_count",
    "possible_mule_receiver",
    "mule_risk_score_v1",
    "mule_alert_v1"
]

combined_df = pd.concat(
    [base_df.reset_index(drop=True), graph_df[graph_feature_cols].reset_index(drop=True)],
    axis=1
)

print("Combined dataset:", combined_df.shape)
combined_df.head()

Combined dataset: (1048575, 28)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,...,risk_score_v1,rule_alert_v1,receiver_in_degree,sender_out_degree,receiver_total_amount,receiver_avg_amount,receiver_high_risk_txn_count,possible_mule_receiver,mule_risk_score_v1,mule_alert_v1
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,...,0,0,1,1,9839.64,9839.640000,0,0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,...,0,0,1,1,1864.28,1864.280000,0,0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,...,45,1,26,1,5653724.70,217450.950000,19,0,50,1
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,...,45,1,27,1,7796257.76,288750.287407,20,0,75,1
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,...,0,0,1,1,11668.14,11668.140000,0,0,0,0


In [4]:
feature_cols_day4 = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "origin_balance_error",
    "dest_balance_error",
    "is_high_risk_type",
    "sender_txn_count",
    "receiver_txn_count",
    "sender_emptied_account",
    "is_large_amount",
    "risk_score_v1",
    "receiver_in_degree",
    "sender_out_degree",
    "receiver_total_amount",
    "receiver_avg_amount",
    "receiver_high_risk_txn_count",
    "possible_mule_receiver",
    "mule_risk_score_v1",
    "mule_alert_v1"
]

X = combined_df[feature_cols_day4]
y = combined_df["isFraud"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1048575, 21)
y shape: (1048575,)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Fraud rate in train:", y_train.mean() * 100)
print("Fraud rate in test:", y_test.mean() * 100)

X_train: (838860, 21)
X_test: (209715, 21)
Fraud rate in train: 0.10895739455928284
Fraud rate in test: 0.10871897575280738


In [6]:
rf_graph_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_graph_model.fit(X_train, y_train)

print("Random Forest with graph features trained successfully")

Random Forest with graph features trained successfully


In [7]:
y_pred_rf_graph = rf_graph_model.predict(X_test)
y_prob_rf_graph = rf_graph_model.predict_proba(X_test)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf_graph))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf_graph))

print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf_graph))
print("PR-AUC:", average_precision_score(y_test, y_prob_rf_graph))

Confusion Matrix:
[[209487      0]
 [     5    223]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    209487
           1       1.00      0.98      0.99       228

    accuracy                           1.00    209715
   macro avg       1.00      0.99      0.99    209715
weighted avg       1.00      1.00      1.00    209715

ROC-AUC: 0.9970865650165122
PR-AUC: 0.9840967311418651


In [8]:
day4_results = pd.DataFrame([
    {
        "model": "Day 2 Random Forest",
        "features": "Transaction + rule features",
        "precision_fraud": 0.77,
        "recall_fraud": 0.98,
        "false_positives": 67,
        "false_negatives": 5,
        "roc_auc": 0.997101,
        "pr_auc": 0.984208
    },
    {
        "model": "Day 4 Random Forest + Graph",
        "features": "Transaction + rule + graph features",
        "precision_fraud": 1.00,
        "recall_fraud": 0.98,
        "false_positives": 0,
        "false_negatives": 5,
        "roc_auc": roc_auc_score(y_test, y_prob_rf_graph),
        "pr_auc": average_precision_score(y_test, y_prob_rf_graph)
    }
])

day4_results

,model,features,precision_fraud,recall_fraud,false_positives,false_negatives,roc_auc,pr_auc
0,Day 2 Random Forest,Transaction + rule features,0.77,0.98,67,5,0.997101,0.984208
1,Day 4 Random Forest + Graph,Transaction + rule + graph features,1.00,0.98,0,5,0.997087,0.984097


In [9]:
feature_importance_day4 = pd.DataFrame({
    "feature": feature_cols_day4,
    "importance": rf_graph_model.feature_importances_
}).sort_values(by="importance", ascending=False)

feature_importance_day4


,feature,importance
5,origin_balance_error,3.357505e-01
12,risk_score_v1,1.337743e-01
1,oldbalanceOrg,1.111013e-01
7,is_high_risk_type,9.682040e-02
2,newbalanceOrig,6.518045e-02
10,sender_emptied_account,5.417983e-02
0,amount,4.025737e-02
17,receiver_high_risk_txn_count,3.749554e-02
3,oldbalanceDest,2.703328e-02
15,receiver_total_amount,2.246394e-02


In [10]:
import joblib

joblib.dump(rf_graph_model, "../models/random_forest_graph_model_day4.joblib")
day4_results.to_csv("../models/day4_graph_model_comparison.csv", index=False)
feature_importance_day4.to_csv("../models/day4_feature_importance.csv", index=False)

print("Day 4 model, comparison, and feature importance saved successfully")

Day 4 model, comparison, and feature importance saved successfully


In [11]:
import os

os.listdir("../models")

['day2_model_comparison.csv',
 'day4_feature_importance.csv',
 'day4_graph_model_comparison.csv',
 'random_forest_fraud_model_day2.joblib',
 'random_forest_graph_model_day4.joblib']